In [111]:
import pandas as pd

# Load saved training and test CSV files from week 3
train = pd.read_csv("data/cleaned_train.csv")
test = pd.read_csv("data/cleaned_test.csv")

print("Training shape: ", train.shape)
print("Test shape: ", test.shape)

Training shape:  (117914, 828)
Test shape:  (12789, 828)


In [112]:
target = "ClosePrice"

city_cols = [
    col for col in train.columns
    if col.startswith("City_grouped_")
]

postal_cols = [
    col for col in train.columns
    if col.startswith("PostalCode_grouped_")
]

county_cols = [
    col for col in train.columns 
    if col.startswith("CountyOrParish_")
]

school_district_cols = [
    col for col in train.columns 
    if col.startswith("SchoolDistrict_")
]

features_sets = {
    "Basic": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet"
        ],

    "With Property Features": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ],

    "With Missing LotSize Flagged": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ],

    "With Location Features (City Only)": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
    ] + city_cols,

    "With Location Features (PostalCode Only)": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ] + postal_cols,

    "With Location Features (County Only)": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ] + county_cols,

    "With All Location Features": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ] + city_cols + postal_cols + county_cols,

    "With Engineered Features": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
        "PropertyAge",
        "BedBathRatio"
    ],

    "With Engineered Features + All Location": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
        "PropertyAge",
        "BedBathRatio"
    ]    + city_cols + postal_cols + county_cols + school_district_cols,

    "With Engineered Features + Unknown YearBuilt + All Location": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
        "PropertyAge",
        "Missing_YearBuilt",
        "BedBathRatio"
    ]    + city_cols + postal_cols + county_cols + school_district_cols

}


In [113]:
for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Ensure no missing values
    print(f"\n{name}")
    print("----------------------")
    missing = X_train.columns[X_train.isna().any()]     # Training Set
    print(missing)
    for col in missing:
        print(col, X_train[col].isna().sum())

    missing = X_test.columns[X_test.isna().any()]       # Test Set
    print(missing)
    for col in missing:
        print(col, X_test[col].isna().sum())

    print("\nTarget Missing")
    print("----------------------")
    print("Train:", y_train.isnull().sum())
    print("Test:", y_test.isnull().sum())


Basic
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With Property Features
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With Missing LotSize Flagged
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With Location Features (City Only)
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With Location Features (PostalCode Only)
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With Location Features (County Only)
----------------------
Index([], dtype='str')
Index([], dtype='str')

Target Missing
----------------------
Train: 0
Test: 0

With All Location Features
----------------------
Index([], dtype='str')

In [114]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

results = []

for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    # print(name, X_train.shape)
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Model
    model = LinearRegression()      # Initialize linear regression model as baseline
    model.fit(X_train, y_train)     # Train model

    # Predictions on target variable
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Compute R^2 scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Save results
    results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Training R^2": round(r2_train, 4),
        "Test R^2": round(r2_test, 4)
    })

    '''# Print results
    print(f"{name}")
    print("-----------------------------------")
    print(f"Training R^2: {r2_train:.4f}")
    print(f"Test R^2: {r2_test:.4f}")
    print()'''

In [115]:
results_df = pd.DataFrame(results)
results_df

,Feature Set,Number of Features,Training R^2,Test R^2
0,Basic,4,0.3639,0.3711
1,With Property Features,8,0.3639,0.3711
2,With Missing LotSize Flagged,9,0.3639,0.3711
3,With Location Features (City Only),209,0.3639,0.3711
4,With Location Features (PostalCode Only),209,0.3639,0.3711
5,With Location Features (County Only),68,0.3639,0.3711
6,With All Location Features,470,0.3639,0.3711
7,With Engineered Features,11,0.4274,0.4407
8,With Engineered Features + All Location,795,0.4274,0.4408
9,With Engineered Features + Unknown YearBuilt +...,796,0.4274,0.4408


Results before removing invalid or extreme values for target (ClosePrice) at the preprocessing step
- Training R^2: 0.0121
- Test R^2: 0.2605

Results after
- Training R^2: 0.3673
- Test R^2: 0.3568

Additional features other than the 4 key variables did not suggest any improvement the prediction preformances for the linear regression model.


Results from Week 6 Updates:
- Training R^2: 0.4274
- Test R^2: 0.441

Adding sample features of PropertyAge, BedBathRatio, and SchoolDistrict improved the prediction performance of the linear regression model.

Compare top coefficients of the features for training set

In [116]:
from sklearn.preprocessing import StandardScaler

features = features_sets["With All Location Features"]

X_train = train[features]
y_train = train[target]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = LinearRegression()
model.fit(X_train_scaled, train[target])

coef_scaled = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_
})

coef_scaled["Abs_Coefficient"] = coef_scaled["Coefficient"].abs()

coef_scaled.sort_values("Abs_Coefficient", ascending=False).head(15)

,Feature,Coefficient,Abs_Coefficient
0,LivingArea,472989.001050,472989.001050
454,CountyOrParish_Santa Clara,353837.806137,353837.806137
439,CountyOrParish_Orange,199578.690429,199578.690429
447,CountyOrParish_San Bernardino,-199209.950794,199209.950794
444,CountyOrParish_Riverside,-196467.968235,196467.968235
162,City_grouped_San Jose,-167752.413844,167752.413844
2,BathroomsTotalInteger,143430.626553,143430.626553
117,City_grouped_Newport Beach,130303.460913,130303.460913
452,CountyOrParish_San Mateo,129362.493095,129362.493095
1,BedroomsTotal,-104735.839138,104735.839138


In [117]:
# Frequency count of each one-hot encoded category
location_groups = {
    "City": city_cols,
    "PostalCode": postal_cols,
    "County": county_cols
}

for name, cols in location_groups.items():
    frequency = train[cols].sum().sort_values(ascending=False)

    print(f"\nTop {name} Frequencies")
    print("--------------------------")
    print(frequency.head(10))


Top City Frequencies
--------------------------
City_grouped_Other          24796.0
City_grouped_Los Angeles     4877.0
City_grouped_San Diego       4048.0
City_grouped_Riverside       1810.0
City_grouped_San Jose        1679.0
City_grouped_Oakland         1515.0
City_grouped_Menifee         1352.0
City_grouped_Long Beach      1207.0
City_grouped_Lancaster       1106.0
City_grouped_Corona          1064.0
dtype: float64

Top PostalCode Frequencies
--------------------------
PostalCode_grouped_Other    60433.0
PostalCode_grouped_92253      818.0
PostalCode_grouped_92345      708.0
PostalCode_grouped_92584      652.0
PostalCode_grouped_92596      611.0
PostalCode_grouped_92592      610.0
PostalCode_grouped_92223      599.0
PostalCode_grouped_92562      554.0
PostalCode_grouped_92211      552.0
PostalCode_grouped_92336      509.0
dtype: float64

Top County Frequencies
--------------------------
CountyOrParish_Los Angeles        29312.0
CountyOrParish_Riverside          17985.0
CountyOrPar

- Another thing to note is that some of the features with relatively large standardized coefficients do not necessarily correspond to the most frequent categories in the dataset. The coefficients here reflects the feature's relationship to ClosePrice.
- LivingArea appears to have the strongest association with predicting ClosePrice.
- Location variables may be meaningful in predicting ClosePrice as well.

Overall Conclusion
- The baseline model generalizes well since R^2 from training set and test set are similar.
- However, the relatively low R^2 suggests underfitting. Further examination is needed on features to include to better capture variations in the prediction of ClosePrice.